# Proyecto Data Lakehouse: Registro Mercantil y Contratación Pública SECOP

**Arquitectura Medallón - Capa Bronze**

---

**Autor:** Jaime Usuga  
**Fecha:** Septiembre 2026  
**Objetivo:** Implementar pipeline de ingesta incremental con datos crudos de empresas colombianas y contratación pública, almacenados en formato Delta Lake con trazabilidad completa.

# Sección 1: Contexto de Negocio

---

## 1.1 Introducción al Dominio

Este proyecto aborda el dominio de **empresas colombianas** y su relación con la **contratación pública estatal**. El análisis integrado de estas dos dimensiones permite comprender patrones económicos, sectores estratégicos y dinámicas de participación empresarial en licitaciones gubernamentales.

### Problemática de Negocio

En Colombia, miles de empresas participan activamente en licitaciones y contratos con entidades del Estado. Sin embargo, la información relevante se encuentra fragmentada en múltiples fuentes:

* **Registro Mercantil:** Contiene información **CORPORATIVA** (identidad legal de la empresa, actividad económica según código CIIU, fecha de constitución, ubicación geográfica, estado de matrícula)
* **SECOP (Sistema Electrónico de Contratación Pública):** Contiene información **TRANSACCIONAL** (contratos adjudicados, montos en pesos colombianos, entidades contratantes, fechas de firma, estado de ejecución)

### Relevancia

Integrar estas fuentes mediante un Data Lakehouse permite:

* 🏢 **Análisis sectorial:** Identificar qué sectores económicos (según clasificación CIIU) concentran mayor participación en contratación estatal
* 💰 **Concentración de contratos:** Detectar empresas con alta recurrencia o alto valor acumulado en contratos públicos
* 📊 **Patrones geográficos:** Analizar la distribución regional de empresas contratistas y su relación con entidades del Estado
* 🔍 **Trazabilidad y transparencia:** Facilitar auditorías, estudios académicos y análisis de políticas públicas

---

## 1.2 Justificación de Fuentes Complementarias

### Campo de Unión

Ambas fuentes se conectan mediante el **NIT (Número de Identificación Tributaria)**, que actúa como llave primaria de integración:

* **En Registro Mercantil:** Campo `numero_identificacion`
* **En SECOP:** Campo `documento_proveedor`

Este campo permite realizar operaciones de JOIN para enriquecer la información transaccional con atributos corporativos.

### Complementariedad

| Aspecto | Registro Mercantil | SECOP |
|---------|-------------------|-------|
| **Tipo de dato** | Corporativo / Maestro | Transaccional / Operacional |
| **Granularidad** | Una fila por empresa | Una fila por contrato |
| **Información única** | Razón social, sector económico (CIIU), fecha de constitución, ubicación | Contratos ganados, valor monetario, entidad contratante, estado del contrato |
| **Pregunta que responde** | ¿Quién es la empresa? ¿A qué se dedica? | ¿Qué contratos ha ganado? ¿Cuánto dinero ha recibido? |
| **Frecuencia de actualización** | Mensual | Diaria |

### Ejemplo de Uso Conjunto

**Pregunta de negocio:** *¿Qué sectores económicos (CIIU) reciben más recursos del Estado a través de contratación pública?*

**Respuesta requiere ambas fuentes:**

1. **Registro Mercantil →** Obtener código CIIU de cada empresa
2. **SECOP →** Obtener valor de contratos por empresa (NIT)
3. **JOIN por NIT →** Relacionar ambas fuentes
4. **Agregación →** Sumar valores de contratos agrupados por código CIIU

**Ejemplo SQL conceptual:**
```sql
SELECT 
    rm.cod_ciiu_act_econ_pri AS codigo_ciiu,
    SUM(sc.valor_del_contrato) AS valor_total_contratos,
    COUNT(DISTINCT sc.documento_proveedor) AS num_empresas
FROM registro_mercantil rm
INNER JOIN secop_contratos sc 
    ON rm.numero_identificacion = sc.documento_proveedor
GROUP BY rm.cod_ciiu_act_econ_pri
ORDER BY valor_total_contratos DESC
LIMIT 10;
```

---

## 1.3 Ficha Técnica - Fuente 1: Registro Mercantil

### Información General

* **Fuente:** Datos Abiertos Colombia (datos.gov.co)
* **Dataset:** Registro Único Empresarial y Social (RUES)
* **URL API:** https://www.datos.gov.co/resource/c82u-588k.json
* **Formato de respuesta:** JSON
* **Volumen aproximado:** ~9 millones de registros históricos
* **Muestra a utilizar:** 100,000 - 200,000 registros recientes (filtrados por fecha de actualización o matrícula)
* **Frecuencia de actualización:** Mensual
* **Licencia:** Datos Abiertos de Colombia (uso libre con atribución)

### Columnas Principales

| Columna | Tipo | Descripción | Ejemplo |
|---------|------|-------------|-------|
| `numero_identificacion` | String | NIT de la empresa (clave de unión) | "900123456" |
| `razon_social` | String | Nombre legal completo de la empresa | "CONSTRUCTORA ABC S.A.S." |
| `cod_ciiu_act_econ_pri` | String | Código de actividad económica principal (clasificación DANE Rev. 4 A.C.) | "4120" (Construcción de edificios) |
| `fecha_matricula` | Date/String | Fecha de registro mercantil inicial | "2015-03-20" |
| `departamento` | String | Ubicación geográfica (departamento) | "ANTIOQUIA" |
| `municipio` | String | Municipio de domicilio principal | "MEDELLÍN" |
| `estado` | String | Estado de la matrícula mercantil | "ACTIVO", "CANCELADO" |
| `tipo_organizacion` | String | Forma jurídica de la empresa | "SOCIEDAD POR ACCIONES SIMPLIFICADA" |

### Observaciones Técnicas

* ⚠️ El campo `numero_identificacion` puede tener valores **nulos** o vacíos en registros históricos o mal formados
* ⚠️ Algunos registros presentan **formato de fecha inconsistente** (ISO 8601 vs formato local); requerirá normalización en capas superiores (Silver)
* ⚠️ La columna `cod_ciiu_act_econ_pri` sigue la **clasificación CIIU Rev. 4 A.C. del DANE**; es un código de 4 dígitos jerárquico
* ✅ La API soporta **paginación** mediante parámetros `$limit` y `$offset` (Socrata Open Data API)
* ✅ Filtrado disponible mediante **SoQL** (Socrata Query Language): `$where=fecha_matricula > '2020-01-01'`

---

## 1.4 Ficha Técnica - Fuente 2: SECOP (Contratación Pública)

### Información General

* **Fuente:** Datos Abiertos Colombia (datos.gov.co)
* **Dataset:** SECOP II - Contratos Electrónicos
* **URL API:** https://www.datos.gov.co/resource/jbjy-vk9h.json
* **Formato de respuesta:** JSON
* **Volumen aproximado:** ~500,000 contratos activos
* **Muestra a utilizar:** 50,000 - 100,000 contratos recientes (filtrados por fecha de firma)
* **Frecuencia de actualización:** Diaria
* **Licencia:** Datos Abiertos de Colombia (uso libre con atribución)

### Columnas Principales

| Columna | Tipo | Descripción | Ejemplo |
|---------|------|-------------|-------|
| `documento_proveedor` | String | NIT del contratista (clave de unión con Registro Mercantil) | "900123456" |
| `nombre_del_proveedor` | String | Nombre comercial o razón social del contratista | "CONSTRUCTORA ABC S.A.S." |
| `valor_del_contrato` | Numeric/String | Monto en pesos colombianos (COP) | "500000000" (500M COP) |
| `nombre_entidad` | String | Entidad estatal que adjudica el contrato | "MINISTERIO DE TRANSPORTE" |
| `departamento_entidad` | String | Ubicación geográfica de la entidad contratante | "CUNDINAMARCA" |
| `fecha_de_firma` | Date/String | Fecha de firma del contrato | "2024-06-15" |
| `estado_contrato` | String | Estado de ejecución del contrato | "VIGENTE", "LIQUIDADO", "TERMINADO" |
| `tipo_de_contrato` | String | Categoría del contrato según objeto | "OBRA", "SUMINISTRO", "CONSULTORÍA" |
| `objeto_del_contrato` | Text | Descripción detallada del objeto contractual | "Construcción de vía terciaria..." |

### Observaciones Técnicas

* ⚠️ El campo `documento_proveedor` puede contener **guiones o puntos** como separadores de dígitos; requerirá limpieza para JOIN con Registro Mercantil
* ⚠️ Valores de contratos pueden ser **$0** (contratos en especie, trueque, o donaciones); requiere tratamiento especial en análisis de valores
* ⚠️ Algunas fechas pueden estar en formato **timestamp ISO 8601** con zona horaria; requerirá extracción de fecha pura
* ✅ La API soporta **paginación** mediante `$limit` y `$offset`
* ✅ Filtrado disponible mediante **SoQL**: `$where=fecha_de_firma > '2023-01-01' AND valor_del_contrato > 10000000`

---

## 1.5 Casos de Uso Potenciales

Una vez integradas las fuentes en la Capa Bronze y disponibles para análisis (posterior transformación en Silver/Gold), se pueden responder preguntas de negocio como:

### 1. Análisis Sectorial de Contratación

**Pregunta:** *¿Qué sectores económicos (según código CIIU) concentran mayor participación en contratación pública?*

**Cómo se responde:**
* JOIN Registro Mercantil (campo `cod_ciiu_act_econ_pri`) con SECOP (campo `documento_proveedor`)
* GROUP BY código CIIU
* SUM(valor_del_contrato) y COUNT(contratos)
* Visualizar los top 10 sectores

**Valor de negocio:** Identificar sectores estratégicos para políticas de fomento empresarial

---

### 2. Concentración de Contratos por Empresa

**Pregunta:** *¿Qué empresas han ganado más contratos (en cantidad y valor) en los últimos 2 años?*

**Cómo se responde:**
* Filtrar SECOP por `fecha_de_firma >= '2024-01-01'`
* COUNT(contratos) y SUM(valor_del_contrato) por `documento_proveedor`
* JOIN con Registro Mercantil para obtener `razon_social` completa
* Ordenar por valor total descendente

**Valor de negocio:** Detectar posibles oligopolios o empresas especializadas en contratación estatal

---

### 3. Análisis Geográfico de Contratistas

**Pregunta:** *¿Qué departamentos tienen empresas con mayor participación en contratación estatal?*

**Cómo se responde:**
* JOIN por NIT entre ambas fuentes
* GROUP BY `departamento` (del Registro Mercantil)
* SUM(valor_del_contrato) de SECOP
* Crear mapa de calor geográfico

**Valor de negocio:** Identificar brechas regionales en capacidad empresarial para contratación pública

---

### 4. Detección de Nuevos Participantes

**Pregunta:** *¿Cuántas empresas creadas en los últimos 3 años ya tienen contratos públicos? ¿En qué sectores?*

**Cómo se responde:**
* Filtrar Registro Mercantil por `fecha_matricula >= '2023-01-01'`
* INNER JOIN con SECOP (solo empresas con contratos)
* Calcular tiempo promedio entre creación y primer contrato
* Agrupar por sector CIIU

**Valor de negocio:** Evaluar barreras de entrada para empresas nuevas en contratación estatal

---

### 5. Análisis de Preferencias por Entidad Contratante

**Pregunta:** *¿Qué tipos de empresas (por CIIU) prefiere contratar cada entidad estatal?*

**Cómo se responde:**
* JOIN completo entre ambas fuentes
* GROUP BY (`nombre_entidad`, `cod_ciiu_act_econ_pri`)
* COUNT(contratos) y AVG(valor_del_contrato)
* Matriz cruzada entidad vs sector

**Valor de negocio:** Entender especialización de proveedores y patrones de demanda por entidad

---

### 6. Trazabilidad y Auditoría

**Pregunta:** *¿Qué empresas con estado "CANCELADO" en Registro Mercantil aún tienen contratos "VIGENTES" en SECOP?*

**Cómo se responde:**
* Filtrar Registro Mercantil por `estado = 'CANCELADO'`
* INNER JOIN con SECOP donde `estado_contrato = 'VIGENTE'`
* Listar casos para revisión

**Valor de negocio:** Detectar inconsistencias para auditorías de transparencia

---

## 📌 Resumen de la Sección 1

Esta sección estableció:

✅ **Dominio claro:** Empresas y contratación pública en Colombia  
✅ **Fuentes complementarias:** Registro Mercantil (corporativo) + SECOP (transaccional)  
✅ **Campo de unión:** NIT (`numero_identificacion` = `documento_proveedor`)  
✅ **Volumen suficiente:** 100K-200K registros (Registro Mercantil) + 50K-100K contratos (SECOP)  
✅ **Casos de uso concretos:** 6 preguntas analíticas demostrables  
✅ **Documentación técnica completa:** URLs, columnas, tipos, observaciones  

**Siguiente paso:** Sección 2 - Diagrama de Arquitectura Bronze

# Sección 2: Diagrama de Arquitectura

---

## 2.1 Flujo de Datos - Arquitectura Bronze

```mermaid
graph TB
    subgraph Fuentes["🌐 FUENTES EXTERNAS - Datos Abiertos Colombia"]
        RM[("Registro Mercantil API<br/>datos.gov.co/ed2q-jk8h<br/>~9M registros")]
        SECOP[("SECOP Contratos API<br/>datos.gov.co/jbjy-vk9h<br/>~500K contratos")]
    end
    
    subgraph Ingesta["⚙️ CAPA DE INGESTA - PySpark"]
        HTTP["HTTP GET paginado<br/>$limit + $offset<br/>(requests)"]  
        PARSE["Parse JSON<br/>→ Spark DataFrame"]
        AUDIT["+ Auditoría<br/>_ingested_at<br/>_source"]
    end
    
    subgraph Bronze["🥉 CAPA BRONZE - Delta Lake + Unity Catalog"]
        T1[("workspace.bronze<br/>bronze_registro_mercantil<br/>Append | Inmutable")]
        T2[("workspace.bronze<br/>bronze_secop_contratos<br/>Append | Inmutable")]
    end
    
    RM -->|JSON paginated| HTTP
    SECOP -->|JSON paginated| HTTP
    HTTP --> PARSE
    PARSE --> AUDIT
    AUDIT -->|mode('append')| T1
    AUDIT -->|mode('append')| T2
    
    style Fuentes fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    style Ingesta fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style Bronze fill:#fce4ec,stroke:#c2185b,stroke-width:2px
```

---

## 2.2 Componentes de la Arquitectura

### 🌐 Fuentes Externas
* **APIs REST de datos.gov.co** (Socrata Open Data)
* Sin autenticación (datos públicos)
* Formato: JSON con paginación (`$limit`, `$offset`)
* Filtrado mediante SoQL: `$where=fecha > '2023-01-01'`

### ⚙️ Capa de Ingesta
* **Tecnología:** PySpark en Databricks
* **Proceso:**
  1. Solicitud HTTP paginada (biblioteca `requests`)
  2. Parsing JSON → Spark DataFrame (`spark.read.json()`)
  3. Agregar campos de auditoría:
     * `_ingested_at`: `current_timestamp()` (timestamp UTC)
     * `_source`: URL completa del endpoint API
  4. Escritura incremental: `mode("append")`

### 🥉 Capa Bronze
* **Almacenamiento:** Delta Lake (ACID, Time Travel, Schema Evolution)
* **Catálogo:** Unity Catalog (`workspace.bronze`)
* **Tablas:**
  * `bronze_registro_mercantil`: Datos corporativos de empresas
  * `bronze_secop_contratos`: Contratos públicos
* **Principio:** Inmutabilidad (columnas originales sin transformación)

---

## 2.3 Decisiones Técnicas Clave

### ¿Por qué PySpark?
* ✅ Escalabilidad para millones de registros
* ✅ Integración nativa con Delta Lake
* ✅ Optimización automática en Databricks

### ¿Por qué Delta Lake?
* ✅ **ACID:** Transacciones atómicas (evita corrupción en fallos)
* ✅ **Time Travel:** Consultar versiones históricas (`VERSION AS OF`)
* ✅ **Schema Evolution:** Agregar columnas sin reescribir datos
* ✅ **Optimización:** `OPTIMIZE` y `Z-ORDER` automáticos

### ¿Por qué modo Append?
* ✅ **Requisito del proyecto:** Carga incremental obligatoria
* ✅ **Eficiencia:** No reprocesar históricos
* ✅ **Auditoría:** Historial completo de cargas

### ¿Por qué campos de auditoría?
* ✅ **Trazabilidad:** Identificar origen y fecha de cada registro
* ✅ **Debugging:** Aislar cargas problemáticas
* ✅ **Gobernanza:** Cumplimiento de estándares de Data Governance

### ¿Por qué inmutabilidad en Bronze?
* ✅ **Reproducibilidad:** Datos crudos siempre disponibles
* ✅ **Debugging:** Rastrear problemas hasta la fuente original
* ✅ **Auditoría:** Garantía de no alteración de datos originales

---

## 📊 Flujo de Ejecución Típico

**Ejemplo: Carga de Registro Mercantil**

```python
# 1. Solicitud HTTP con paginación
url = "https://www.datos.gov.co/resource/ed2q-jk8h.json?$limit=10000&$offset=0"
response = requests.get(url)

# 2. Parse JSON → DataFrame
df = spark.read.json(sc.parallelize([response.text]))

# 3. Agregar auditoría
df_audit = df.withColumn("_ingested_at", current_timestamp()) \
              .withColumn("_source", lit(url))

# 4. Escribir a Delta (append)
df_audit.write.format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.bronze_registro_mercantil")
```

---

## 🔄 Carga Incremental - Estrategia

| Ejecución | Acción | Registros Nuevos | Total Acumulado |
|-----------|--------|------------------|------------------|
| **1ª carga** | Descarga inicial (ej: 100K registros) | 100,000 | 100,000 |
| **2ª carga** | Solo registros nuevos (filtro por fecha) | 5,000 | 105,000 |
| **3ª carga** | Solo registros nuevos | 3,000 | 108,000 |

**Ventaja:** No reprocesar los 100K iniciales en cada carga.

---

## 📌 Resumen de la Sección 2

✅ **Arquitectura visual:** Flujo completo desde APIs hasta Delta Lake  
✅ **3 capas definidas:** Fuentes → Ingesta → Bronze  
✅ **Tecnologías justificadas:** PySpark, Delta Lake, Unity Catalog  
✅ **Principios aplicados:** Inmutabilidad, Auditoría, Carga Incremental  
✅ **Estrategia clara:** Append mode + campos _ingested_at/_source  

**Siguiente paso:** Sección 3 - Implementación del Pipeline (Código PySpark)

# Sección 3: Pipeline de Ingesta - Implementación

---

Esta sección contiene el código PySpark funcional que implementa el pipeline de ingesta para ambas fuentes de datos.

## Objetivos de la Sección 3

✅ **Descarga paginada** desde APIs REST de datos.gov.co  
✅ **Transformación** a Spark DataFrame  
✅ **Auditoría** con campos `_ingested_at` y `_source`  
✅ **Persistencia** en Delta Lake (Unity Catalog)  
✅ **Carga incremental** con `mode("append")`  
✅ **Inmutabilidad** de columnas originales  

---

## Estructura del Pipeline

**3.1 Pipeline Fuente 1: Registro Mercantil**
- Configuración de parámetros
- Función de descarga paginada
- Transformación y auditoría
- Escritura a Delta Lake

**3.2 Verificación de Carga**
- Conteo de registros
- Visualización de muestra

**3.3 Pipeline Fuente 2: SECOP Contratos** (siguiente etapa)

---

**Nota:** Ejecutar las celdas en orden secuencial.

In [0]:
# ============================================================================
# PIPELINE DE INGESTA: REGISTRO MERCANTIL
# ============================================================================
# Descarga datos del Registro Único Empresarial y Social (RUES) desde la API
# de Datos Abiertos Colombia y los almacena en Delta Lake (Capa Bronze)
# ============================================================================

# ----------------------------------------------------------------------------
# 1. IMPORTACIONES
# ----------------------------------------------------------------------------
import requests
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import StructType, StructField, StringType

print("✅ Importaciones cargadas correctamente")

# ----------------------------------------------------------------------------
# 2. CONFIGURACIÓN DE PARÁMETROS
# ----------------------------------------------------------------------------

# URL base del API de Registro Mercantil (RUES)
API_URL = "https://www.datos.gov.co/resource/c82u-588k.json"

# Parámetros de paginación
LIMIT = 5000          # Registros por página (ajustable según rendimiento API)
MAX_RECORDS = 20000   # Total de registros a descargar (para demostración)
                      # Para producción, aumentar a 100000-200000

# Tabla destino en Unity Catalog
TARGET_TABLE = "workspace.bronze.bronze_registro_mercantil"

# 🆕 CÁLCULO AUTOMÁTICO DE OFFSET (CARGA INCREMENTAL INTELIGENTE)
# Verifica cuántos registros ya existen y empieza desde ahí
print(f"\n🔍 Verificando estado de la tabla {TARGET_TABLE}...")
try:
    # Contar registros existentes en la tabla
    conteo_actual = spark.sql(f"SELECT COUNT(*) as total FROM {TARGET_TABLE}").collect()[0]['total']
    OFFSET_START = conteo_actual  # Empezar después del último registro
    print(f"✅ Tabla existente: {conteo_actual:,} registros")
    print(f"✅ Próxima descarga: desde registro {OFFSET_START:,}")
except Exception as e:
    # Si la tabla no existe (primera carga), empezar en 0
    OFFSET_START = 0
    print(f"✅ Primera carga: tabla no existe, empezando desde registro 0")

print(f"\n📊 Configuración:")
print(f"   - API: {API_URL}")
print(f"   - Offset inicial: {OFFSET_START:,}")
print(f"   - Registros por página: {LIMIT:,}")
print(f"   - Registros a descargar: {MAX_RECORDS:,}")
print(f"   - Tabla destino: {TARGET_TABLE}")

# ----------------------------------------------------------------------------
# 3. FUNCIÓN DE DESCARGA PAGINADA
# ----------------------------------------------------------------------------

def descargar_datos_paginados(url, limit, max_records):
    """
    Descarga datos de la API usando paginación.
    
    Parámetros:
    - url: URL base del API
    - limit: Registros por página
    - max_records: Total máximo de registros a descargar
    
    Retorna:
    - Lista de diccionarios con los datos descargados
    """
    datos_completos = []
    offset = OFFSET_START
    pagina = 1
    
    print("\n🔄 Iniciando descarga paginada...")
    
    while len(datos_completos) < max_records:
        # Construir URL con parámetros de paginación
        url_paginada = f"{url}?$limit={limit}&$offset={offset}"
        
        try:
            # Realizar solicitud HTTP GET
            response = requests.get(url_paginada, timeout=30)
            response.raise_for_status()  # Lanza excepción si status != 200
            
            # Parsear respuesta JSON
            datos_pagina = response.json()
            
            # Si no hay más datos, terminar
            if not datos_pagina or len(datos_pagina) == 0:
                print(f"   ℹ️  Página {pagina}: Sin más datos disponibles")
                break
            
            # Agregar datos de esta página
            datos_completos.extend(datos_pagina)
            
            print(f"   ✅ Página {pagina}: {len(datos_pagina)} registros descargados | Total acumulado: {len(datos_completos)}")
            
            # Incrementar offset para siguiente página
            offset += limit
            pagina += 1
            
            # Detener si alcanzamos el límite
            if len(datos_completos) >= max_records:
                print(f"   ⚠️  Límite de {max_records} registros alcanzado")
                datos_completos = datos_completos[:max_records]  # Truncar si excede
                break
                
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Error en página {pagina}: {str(e)}")
            break
    
    print(f"\n✅ Descarga completada: {len(datos_completos)} registros totales")
    return datos_completos

# ----------------------------------------------------------------------------
# 4. EJECUCIÓN DE DESCARGA
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("FASE 1: DESCARGA DE DATOS")
print("="*70)

datos_raw = descargar_datos_paginados(API_URL, LIMIT, MAX_RECORDS)

if len(datos_raw) == 0:
    raise Exception("❌ No se descargaron datos. Verifica la conectividad del API.")

print(f"\n✅ {len(datos_raw)} registros listos para transformación")

# ----------------------------------------------------------------------------
# 5. CONVERSIÓN A SPARK DATAFRAME
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("FASE 2: CONVERSIÓN A SPARK DATAFRAME")
print("="*70)

# Crear DataFrame directamente desde lista de diccionarios
# (Compatible con Serverless Compute - no requiere sc.parallelize)
df_raw = spark.createDataFrame(datos_raw)

print(f"✅ DataFrame creado con {df_raw.count()} filas y {len(df_raw.columns)} columnas")
print(f"\n📋 Schema inferido:")
df_raw.printSchema()

# ----------------------------------------------------------------------------
# 6. AGREGAR CAMPOS DE AUDITORÍA
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("FASE 3: AGREGAR CAMPOS DE AUDITORÍA")
print("="*70)

# Agregar columnas de trazabilidad
df_auditado = df_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source", lit(API_URL))

print("✅ Campos de auditoría agregados:")
print("   - _ingested_at: Timestamp de carga (UTC)")
print("   - _source: URL de origen de los datos")

print(f"\n📊 Columnas disponibles en el DataFrame:")
print(f"   {df_auditado.columns}")

print(f"\n📊 Muestra de datos con auditoría (primeras 5 filas):")
df_auditado.show(5, truncate=False)

# ----------------------------------------------------------------------------
# 7. ESCRITURA A DELTA LAKE (CAPA BRONZE)
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("FASE 4: ESCRITURA A DELTA LAKE")
print("="*70)

print(f"🔄 Escribiendo datos a {TARGET_TABLE}...")
print(f"   Modo: APPEND (incremental)")
print(f"   Formato: Delta Lake")

try:
    df_auditado.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(TARGET_TABLE)
    
    print(f"\n✅ ¡CARGA EXITOSA!")
    print(f"   Tabla: {TARGET_TABLE}")
    print(f"   Registros escritos: {df_auditado.count()}")
    print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except Exception as e:
    print(f"\n❌ ERROR al escribir a Delta Lake: {str(e)}")
    raise

# ----------------------------------------------------------------------------
# 8. RESUMEN FINAL
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("RESUMEN DE EJECUCIÓN")
print("="*70)
print(f"✅ Pipeline ejecutado correctamente")
print(f"📊 Estadísticas:")
print(f"   - Registros descargados: {len(datos_raw)}")
print(f"   - Registros escritos: {df_auditado.count()}")
print(f"   - Columnas originales: {len(df_raw.columns)}")
print(f"   - Columnas con auditoría: {len(df_auditado.columns)}")
print(f"   - Tabla destino: {TARGET_TABLE}")
print("\n🎯 Próximo paso: Ejecutar la celda de verificación SQL")

In [0]:
%sql
-- ============================================================================
-- VERIFICACIÓN DE CARGA: REGISTRO MERCANTIL
-- ============================================================================
-- Consultas para validar que los datos se cargaron correctamente
-- en la tabla Bronze
-- ============================================================================

-- 1. CONTEO TOTAL DE REGISTROS
SELECT 
    '📊 Total de registros' AS metrica,
    CAST(COUNT(*) AS STRING) AS valor
FROM workspace.bronze.bronze_registro_mercantil

UNION ALL

-- 2. CONTEO DE REGISTROS ÚNICOS POR NIT
SELECT 
    '🔑 NITs únicos' AS metrica,
    CAST(COUNT(DISTINCT numero_identificacion) AS STRING) AS valor
FROM workspace.bronze.bronze_registro_mercantil

UNION ALL

-- 3. REGISTROS CON AUDITORÍA
SELECT 
    '✅ Registros con auditoría' AS metrica,
    CAST(COUNT(*) AS STRING) AS valor
FROM workspace.bronze.bronze_registro_mercantil
WHERE _ingested_at IS NOT NULL

UNION ALL

-- 4. FECHA DE ÚLTIMA CARGA
SELECT 
    '📅 Última carga' AS metrica,
    CAST(MAX(_ingested_at) AS STRING) AS valor
FROM workspace.bronze.bronze_registro_mercantil;